# Misconception-Channel Modelling and Comparison

Loads the saved misconception labels (from notebook 03), joins them to the
correctness/KC data, and compares:

- **Baseline**: correctness-only BKT.
- **Design 1 (pooled)**: + misconception emission, pooled across families.
- **Design 2a (family)**: + family-conditioned misconception emission.

each at **binary** and **trinary** granularity. Reports AUC / Acc / log-lik /
Brier overall, against the majority-class floor, in one table.

No API calls here; this is pure modelling on the saved labels.


## Setup

In [1]:
import os
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "data" / "annotated").exists():
        os.chdir(_c); break
import sys
sys.path.insert(0, str(Path.cwd() / "extension"))
print("cwd:", os.getcwd())


cwd: /Users/tandon.utsav2/Desktop/Experiment_1


In [2]:
from ast import literal_eval
import pandas as pd, numpy as np, time
from myext import filtering, bkt, bkt_design1, bkt_design2, bkt_mc_common, labels_io

CONV = {c: literal_eval for c in ["annotation", "dialogue"]}
def load(split):
    df = pd.read_csv(f"data/annotated/mathdial_{split}_atc.csv", converters=CONV)
    df, _ = filtering.drop_failed_annotations(df); df.index = df["index"]
    return df
def turn_table(df):
    rows=[]
    for didx, ann in zip(df.index, df["annotation"]):
        if not isinstance(ann, dict): continue
        for tk, info in ann.items():
            if not isinstance(info, dict) or "correct" not in info or "kcs" not in info: continue
            c = 1 if info["correct"] in (True,1,"true","correct") else 0
            for kc in info["kcs"]: rows.append((didx, tk, c, kc))
    return pd.DataFrame(rows, columns=["dialogue_idx","turn","correct","kc"])

train = load("train"); test = load("test")
train_long = turn_table(train); test_long = turn_table(test)

# degenerate-KC filter (same as the baseline notebook)
train_f, test_f, fres = filtering.filter_splits(train_long, test_long,
                                                drop_no_variation=True, min_count=1)
print(fres.summary())

# majority-class floor
per_turn = test_long.groupby(["dialogue_idx","turn"])["correct"].first()
floor = max(per_turn.mean(), 1 - per_turn.mean())
print(f"majority-class floor (test Acc): {floor:.3f}")


KCs: 143 -> 124 (dropped 19)
observations: 29478 -> 29447 (removed 31, 0.11%)
drop reasons: no_variation=19
majority-class floor (test Acc): 0.588


## Load and join the saved misconception labels

From `data/misconception/` (written by notebook 03). The per-turn label
broadcasts across the KCs each turn invokes.


In [3]:
train_lab = labels_io.load_labels("train")
test_lab = labels_io.load_labels("test")
print("train labels:", len(train_lab), "| test labels:", len(test_lab))
print("train misc dist:", train_lab["misc"].value_counts(normalize=True).round(3).to_dict())

train_mc = labels_io.join_misc(train_f, train_lab)
test_mc  = labels_io.join_misc(test_long, test_lab)
print("joined train rows:", len(train_mc), "| any unlabelled misc:",
      (train_mc["misc"] == "not_evidenced").mean().round(3), "not_evidenced frac")


train labels: 13219 | test labels: 3498
train misc dist: {'present': 0.526, 'absent': 0.322, 'not_evidenced': 0.152}
joined train rows: 29447 | any unlabelled misc: 0.13 not_evidenced frac


## Fit all models

Baseline plus the four augmented variants (2 designs x 2 granularities).
Each fit is pure-Python EM; the family design is the slowest. Cached nowhere,
so this runs the fits fresh each time (a few minutes total).


In [4]:
N_RESTARTS = 3
results = {}

t0 = time.time()
fb = bkt.fit_bkt(train_f, n_restarts=N_RESTARTS, verbose=False)
ov, fn, _ = bkt.evaluate(fb, test_long)
results["baseline (correctness only)"] = ov
print(f"baseline done ({time.time()-t0:.0f}s)")

fit_fns = {"D1 pooled": bkt_design1.fit_design1,
           "D2a family": bkt_design2.fit_design2}
for label, fit_fn in fit_fns.items():
    for gran in ["binary", "trinary"]:
        t0 = time.time()
        fm = fit_fn(train_mc, gran, n_restarts=N_RESTARTS, verbose=False)
        ovm, fnm, _ = bkt_mc_common.evaluate_mc(fm, test_mc)
        name = f"{label} + {gran}"
        results[name] = ovm
        print(f"{name} done ({time.time()-t0:.0f}s)")


baseline done (70s)
D1 pooled + binary done (47s)
D1 pooled + trinary done (37s)
D2a family + binary done (46s)
D2a family + trinary done (45s)


## Comparison table

In [5]:
rows = []
for name, m in results.items():
    rows.append({"model": name, "Acc": round(m.accuracy,4), "AUC": round(m.auc,4),
                 "LogLik": round(m.log_likelihood,4), "Brier": round(m.brier,4)})
tbl = pd.DataFrame(rows)
tbl["floor_Acc"] = round(floor,3)
# delta AUC vs baseline
base_auc = results["baseline (correctness only)"].auc
tbl["dAUC_vs_base"] = (tbl["AUC"] - base_auc).round(4)
print(tbl.to_string(index=False))


                      model    Acc    AUC  LogLik  Brier  floor_Acc  dAUC_vs_base
baseline (correctness only) 0.6135 0.6271 -0.6503 0.2291      0.588        0.0000
         D1 pooled + binary 0.5792 0.6040 -0.6959 0.2466      0.588       -0.0231
        D1 pooled + trinary 0.6002 0.6193 -0.6749 0.2386      0.588       -0.0078
        D2a family + binary 0.5829 0.6060 -0.6980 0.2469      0.588       -0.0211
       D2a family + trinary 0.5984 0.6201 -0.6773 0.2387      0.588       -0.0070


### Reading the table

- **dAUC_vs_base** is the headline: does adding the misconception channel raise
  AUC over correctness-only BKT, and by how much?
- Compare **binary vs trinary** within each design: does modelling the
  not-evidenced turns as no-update (trinary) beat folding them into absent
  (binary)?
- Compare **D1 pooled vs D2a family**: does family-conditioning the
  misconception emission help, or is the data too sparse for it (in which case
  pooled wins)?
- All should clear the majority-class floor on AUC. Small deltas are expected;
  the channel is a refinement on top of correctness, not a replacement.

These are proof-of-concept results on simulated-student labels; interpret the
direction and sign of the effect rather than absolute magnitudes.


## Optional: paired significance

If you want a paired test on whether the channel's per-turn predictions differ
from baseline, run McNemar or a bootstrap over per-turn correctness predictions.
Stub below compares the best augmented model's predictions to baseline.


In [6]:
# bootstrap CI on the AUC difference (best augmented vs baseline)
from sklearn.metrics import roc_auc_score
best_name = max((n for n in results if n!="baseline (correctness only)"),
                key=lambda n: results[n].auc)
print("best augmented model:", best_name)

base_pred = fb.predict_long(test_long)[["dialogue_idx","turn","kc","correct"]].copy()
base_pred["pred_base"] = bkt.FittedBKT(fb.per_skill, fb.fallback).predict_long(test_long)["pred"].values
# refit best for its predictions
g = "binary" if "binary" in best_name else "trinary"
fit_fn = bkt_design1.fit_design1 if "pooled" in best_name else bkt_design2.fit_design2
fm_best = fit_fn(train_mc, g, n_restarts=N_RESTARTS, verbose=False)
mc_pred = bkt_mc_common.evaluate_mc(fm_best, test_mc)[2]

y = base_pred["correct"].to_numpy()
pb = base_pred["pred_base"].to_numpy()
pm = mc_pred.sort_values(["dialogue_idx","kc","turn"])["pred"].to_numpy()[:len(y)]

rng = np.random.default_rng(0); diffs=[]
for _ in range(1000):
    idx = rng.integers(0, len(y), len(y))
    try:
        diffs.append(roc_auc_score(y[idx], pm[idx]) - roc_auc_score(y[idx], pb[idx]))
    except ValueError:
        pass
diffs = np.array(diffs)
print(f"AUC difference (best - baseline): {diffs.mean():.4f} "
      f"[{np.percentile(diffs,2.5):.4f}, {np.percentile(diffs,97.5):.4f}] (95% bootstrap CI)")
print("CI excludes 0:", not (np.percentile(diffs,2.5) < 0 < np.percentile(diffs,97.5)))


best augmented model: D2a family + trinary


KeyboardInterrupt: 

## Diagnostics: why does the channel behave as it does?

Given that the labels are highly informative (large `P(correct|misc)` gap) yet
the channel does not help, these two analyses explain why.

**Overconfidence**: if the misconception is nearly a copy of correctness, the
emission double-counts it and the belief becomes overconfident. Expect the
augmented model to be more extreme and worse calibrated (higher ECE, Brier).

**Off-diagonal**: the channel can only add information correctness lacks on
turns where misconception and correctness DISAGREE (present-but-correct,
absent-but-incorrect). This splits performance by agree/disagree to locate
where the channel helps or hurts.


In [7]:
from myext import diagnostics

# baseline and best-augmented predictions, aligned
base_pred = fb.predict_long(test_long)
# pick the best augmented model by AUC (refit it for predictions)
best_name = max((n for n in results if n!='baseline (correctness only)'),
                key=lambda n: results[n].auc)
g = 'binary' if 'binary' in best_name else 'trinary'
fit_fn = bkt_design1.fit_design1 if 'pooled' in best_name else bkt_design2.fit_design2
fm_best = fit_fn(train_mc, g, n_restarts=N_RESTARTS, verbose=False)
aug_pred = fm_best.predict_long(test_mc)
print('diagnosing:', best_name)

# 1) overconfidence (align both prediction frames the same way)
bp = base_pred.sort_values(['dialogue_idx','kc','turn']).reset_index(drop=True)
ap = aug_pred.sort_values(['dialogue_idx','kc','turn']).reset_index(drop=True)
print('\n=== OVERCONFIDENCE (aug more extreme / worse calibrated => double-counting) ===')
oc = diagnostics.overconfidence_report(bp, ap)
for k, v in oc.items():
    print(f'  {k}: {v:.4f}')

# 2) off-diagonal: where misconception and correctness disagree
print('\n=== OFF-DIAGONAL (does the channel help where it disagrees with correctness?) ===')
od = diagnostics.offdiagonal_report(aug_pred, base_pred, test_mc)
print(od.to_string(index=False))


diagnosing: D2a family + trinary

=== OVERCONFIDENCE (aug more extreme / worse calibrated => double-counting) ===
  base_extremity: 0.1075
  aug_extremity: 0.1520
  base_ece: 0.0272
  aug_ece: 0.0633
  base_brier: 0.2291
  aug_brier: 0.2387
  frac_aug_more_extreme: 0.7524

=== OFF-DIAGONAL (does the channel help where it disagrees with correctness?) ===
       subset    n  frac_correct  base_AUC  aug_AUC  base_acc  aug_acc    dAUC
        agree 2828         0.274    0.6482   0.6757    0.6970   0.7104  0.0275
     disagree 2161         0.536    0.5730   0.5281    0.5160   0.4817 -0.0449
not_evidenced 2457         0.457    0.6330   0.6064    0.6032   0.5722 -0.0266
